In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'prices_round_3_day_0.csv',
    'prices_round_3_day_1.csv',
    'prices_round_3_day_2.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Parameters
ALPHA = 0.01
WINDOW = 50
Z_THRESHOLD = 2.3
products = ["HYDROGEL_PACK"]

days = [0, 1, 2]

# Assuming df_total exists in your environment
for day in days:
    subset = df_total[(df_total['product'].isin(products)) & (df_total['day'] == day)].copy()
    
    if subset.empty: 
        continue

    # 1. Clean data and handle zeros
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
    
    for product in products:
        prod_data = subset[subset['product'] == product].copy().sort_values('timestamp')
        
        # --- Optimized Signal Calculation ---
        # Use Pandas built-ins for EWMA and Rolling Std (much more robust)
        prod_data['ewma_mean'] = prod_data['mid_price'].ewm(alpha=ALPHA, adjust=False).mean()
        
        # Calculate rolling volatility of the price relative to the mean
        # We use the rolling standard deviation of the price to normalize the Z-Score
        prod_data['rolling_std'] = prod_data['mid_price'].rolling(window=WINDOW).std()
        
        # Z-Score: (Current - Mean) / Volatility
        prod_data['z_score'] = (prod_data['mid_price'] - prod_data['ewma_mean']) / prod_data['rolling_std']
        
        # Fill NaNs from the initial window so Plotly doesn't skip the first 50 points
        prod_data['z_score'] = prod_data['z_score'].fillna(0)

        # --- Enhanced Graphing ---
        fig = make_subplots(
            rows=2, cols=1, 
            shared_xaxes=True, 
            vertical_spacing=0.08, 
            row_heights=[0.7, 0.3],
            subplot_titles=(f"{product} Price Heatmap", "Z-Score Signal")
        )

        # Row 1: The "Heatmap" Overlay
        # Trace 1: The background line (thin and light)
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='lines', name='Price Path',
            line=dict(color='rgba(150, 150, 150, 0.3)', width=1)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['ewma_mean'],
            mode='lines', name='Price Path',
        ), row=1, col=1)

        # Trace 2: The colored markers (The actual signal)
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='markers',
            name='Z-Score Color',
            marker=dict(
                size=6,
                color=prod_data['z_score'],
                colorscale='RdBu', 
                reversescale=True,  # Red = High/Overbought, Blue = Low/Oversold
                cmid=0,             # Ensure 0 is the neutral center color
                cmin=-Z_THRESHOLD,  # Cap color intensity at your threshold
                cmax=Z_THRESHOLD,
                showscale=True,
                colorbar=dict(title="Z-Score", x=1.02, len=0.7)
            ),
            hovertext=[f"Price: {p:.2f}<br>Z: {z:.2f}" for p, z in zip(prod_data['mid_price'], prod_data['z_score'])],
            hoverinfo="text+x"
        ), row=1, col=1)

        # Row 2: Standard Z-Score Line
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['z_score'],
            name='Z-Score Value', 
            line=dict(color='black', width=1.2)
        ), row=2, col=1)

        # Threshold lines
        for val in [Z_THRESHOLD, -Z_THRESHOLD]:
            fig.add_hline(y=val, line_dash="dash", line_color="red", row=2, col=1)

        fig.update_layout(
            height=800, 
            template='plotly_white', 
            title_text=f"Market Analysis: {product} (Day {day})",
            showlegend=False
        )
        
        fig.show()

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Configuration ---
# Kalman Gain Control: 
# Increase R to make the Local Mean "stiffer" (slower to move)
# Increase Q to make the Local Mean "faster" (track spikes more closely)
Q_PROCESS = 1e-5 
R_MEASURE = 0.05 

WINDOW = 100        # For volatility calculation
Z_THRESHOLD = 2.0
products = ["HYDROGEL_PACK"]

def apply_kalman_local_mean(series, q=1e-5, r=0.01, global_mean=10000):
    """Tracks the 'Local' consensus price, ignoring micro-noise."""
    means = []
    curr_est = series.iloc[0] if not series.empty else global_mean
    error_est = 1.0
    for val in series:
        # Prediction step
        error_est += q
        # Update step
        kalman_gain = error_est / (error_est + r)
        curr_est = curr_est + kalman_gain * (val - curr_est)
        error_est = (1 - kalman_gain) * error_est
        means.append(curr_est)
    return pd.Series(means, index=series.index)

for day in [0, 1, 2]:
    subset = df_total[(df_total['product'].isin(products)) & (df_total['day'] == day)].copy()
    if subset.empty: continue
    
    # Cleaning
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan).ffill()
    
    for product in products:
        prod_data = subset[subset['product'] == product].copy().sort_values('timestamp')

        global_mean = prod_data['mid_price'].mean() # Could also be a fixed value like 10k
        
        # 1. Local Anchor (The Kalman Filter)
        # This captures the "smaller mean reversions" around spikes
        prod_data['local_mean'] = apply_kalman_local_mean(prod_data['mid_price'], Q_PROCESS, R_MEASURE, global_mean)
        
        # 2. Volatility (Standard Deviation around the Local Mean)
        # We use a floor (1e-2) to prevent the Z-score from exploding in flat markets
        prod_data['rolling_std'] = prod_data['mid_price'].rolling(window=WINDOW).std().replace(0, np.nan).ffill().fillna(1.0)
        
        # 3. The Signal: Deviation from Local Mean
        prod_data['z_score'] = (prod_data['mid_price'] - prod_data['local_mean']) / prod_data['rolling_std']
        
        # 4. Global Bias: How far is our "Local Consensus" from the 10k "True North"?
        prod_data['global_bias'] = prod_data['local_mean'] - global_mean

        # --- Visualization ---
        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, 
            vertical_spacing=0.05, row_heights=[0.7, 0.3],
            subplot_titles=(f"{product} Local vs Global Mean", "Reversion Signal (Z-Score)")
        )

        # Main Plot: Price & Means
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['mid_price'], 
                                 line=dict(color='rgba(200,200,200,0.4)', width=1), name='Market Price'), row=1, col=1)
        
        # Global  Anchor
        fig.add_hline(y=global_mean, line_dash="dot", line_color="black", annotation_text=f"Global Anchor {global_mean:.2f} ", row=1, col=1)
        
        # Local Kalman Mean
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['local_mean'], 
                                 line=dict(color='orange', width=2), name='Local Mean (Kalman)'), row=1, col=1)

        # Heatmap Markers
        fig.add_trace(go.Scatter(
            x=prod_data['timestamp'], y=prod_data['mid_price'],
            mode='markers', name='Trade Signal',
            marker=dict(
                size=5, color=prod_data['z_score'], colorscale='RdBu', reversescale=True,
                cmid=0, cmin=-Z_THRESHOLD, cmax=Z_THRESHOLD, showscale=True
            )
        ), row=1, col=1)

        # Z-Score Subplot
        fig.add_trace(go.Scatter(x=prod_data['timestamp'], y=prod_data['z_score'], 
                                 line=dict(color='black', width=1), name='Z-Score'), row=2, col=1)
        
        for v in [Z_THRESHOLD, -Z_THRESHOLD]:
            fig.add_hline(y=v, line_dash="dash", line_color="red", row=2, col=1)

        fig.update_layout(height=900, template='plotly_white', title_text=f"Hybrid Analysis: {product} (Day {day})", showlegend=False)
        fig.show()